# Basics &mdash; Sets, Cardinality, and the Infinite Sets

**Concept 1 of the Basics decomposition:** *Sets, Cardinality, and the Infinite Sets $Nat$, $Int$, $Real$*

A finite cardinality is a member of $Nat$; an infinite one is not &mdash; $\aleph_0$ versus $2^{\aleph_0}$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Sets-And-Cardinality/Concept-Sets-And-Cardinality.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A **set** is an unordered collection with no duplicates. Its **cardinality** $|S|$ is
the number of members.

The distinction the book needs later: a **finite** cardinality *is a member of $Nat$*;
an **infinite** one is not. $|Nat| = \aleph_0$ is not a natural number, it is a new
kind of object.

Three infinite sets recur throughout: $Nat = \{0,1,2,\ldots\}$, $Int$, and $Real$.
The first two have cardinality $\aleph_0$ &mdash; they can be put in bijection with each
other. $Real$ cannot, and $|Real| = 2^{\aleph_0} > \aleph_0$.

That strict inequality is the whole content of Appendix C, and it is why
**non-RE languages must exist**: countably many machines, uncountably many languages.

## 2. Definitions

### Sets in Python, and cardinality

In [ ]:
A = {1, 2, 3}
B = {3, 2, 1, 1, 2}          # duplicates and order do not matter
print("A =", A, "  B =", B, "  equal?", A == B)
print("|A| =", len(A))
assert A == B and len(A) == 3

### Enumerating $Nat$, $Int$ and the rationals &mdash; all $\aleph_0$

In [ ]:
def nat(n):
    return list(range(n))

def integers(n):
    # 0, 1, -1, 2, -2, ... -- a bijection Nat -> Int
    out = [0]
    k = 1
    while len(out) < n:
        out += [k, -k]; k += 1
    return out[:n]

def rationals(n):
    # walk the diagonals of the p/q grid, skipping repeats
    from math import gcd
    out, d = [], 2
    while len(out) < n:
        for p in range(1, d):
            q = d - p
            if gcd(p, q) == 1: out.append((p, q))
            if len(out) == n: break
        d += 1
    return out

<!-- nav-strip -->

---

[**Basics** index](https://github.com/ganeshutah/Jove/blob/master/Basics/README.md) &nbsp;&middot;&nbsp; [Basics&nbsp;2.&nbsp;Set Builder Notation](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Set-Builder-Notation/Concept-Set-Builder-Notation.ipynb)&nbsp;&rarr;

---

## 3. Tests

$Int$ is the same size as $Nat$: here is the bijection.

In [ ]:
print("first 11 integers enumerated :", integers(11))
xs = integers(400)
print("no duplicates? ", len(xs) == len(set(xs)))
assert len(xs) == len(set(xs))
print("every integer in -50..50 appears? ",
      set(range(-50, 51)) <= set(integers(1000)))
assert set(range(-50, 51)) <= set(integers(1000))

So does the set of positive rationals &mdash; the classic surprise.

In [ ]:
print("first 10 rationals :", rationals(10))
rs = rationals(300)
assert len(rs) == len(set(rs))
print("\n'more' rationals than integers is FALSE -- both are aleph_0.")

**$Real$ is strictly bigger**, by Cantor's diagonal.

In [ ]:
# a finite rehearsal: any LIST of binary expansions misses one
listing = ['0101010101', '1100110011', '0011001100', '1111000011',
           '1010101010', '0000111100', '1001001001', '0110110110',
           '1110001110', '0101101011']
diag = ''.join('1' if row[i] == '0' else '0' for i, row in enumerate(listing))
print("the listing :")
for i, r in enumerate(listing): print("   %d: %s" % (i, r))
print("\ndiagonal-flipped :", diag)
for i, r in enumerate(listing):
    assert diag[i] != r[i]
print("differs from row i at position i, for every i -- so it is not in the list")

The consequence the book cares about.

In [ ]:
print("Turing machines : each is a finite string over a finite alphabet")
print("                  -> countably many, aleph_0")
print("languages       : each is a SUBSET of the countable set Sigma*")
print("                  -> 2^aleph_0 of them")
print()
print("2^aleph_0 > aleph_0, so almost every language has NO machine.")
print("That is Appendix C, and Chapter 14 Concept 11.")

## 4. Exercises


1. Give the bijection $Nat \to Int$ as a formula rather than a listing.
2. Why does the diagonal argument fail if you try it on the rationals?
3. Is the set of all **finite** subsets of $Nat$ countable? All subsets?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Basics/Concept-Sets-And-Cardinality')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')